# 5. Feature Selection Study

This notebook investigates how the **fraction of features** used during training affects model performance (RMSE).

Features are selected symmetrically from both ends of the feature array:
- `selected_count = max(1, round(total_features / 2 * fraction + 0.1))`
- indices = first `N` features + last `N` features

Sections:
1. [Hyperparameter Tuning with Feature Subsets](#1-hyperparameter-tuning-with-feature-subsets)
2. [RMSE vs Feature Fraction Analysis (LightGBM)](#2-rmse-vs-feature-fraction-analysis-lightgbm)

In [ ]:
import json
import re
import warnings

import mlflow
import optuna
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

from utils.tuning import (
    tune_hyperparameters,
    get_hyperparams_getter_function_by_model_name,
    prepare_hyperparams,
)
from utils.config import load_config
from utils.shared import get_model_by_name

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:
config = load_config()

FEATURES_PERCENTAGES = [0.3, 0.5, 0.7, 0.9, 1.0]
FS_EXPERIMENT_NAME = "tuning_fs_1"

# Publication-quality plot style
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "lines.linewidth": 2,
        "lines.markersize": 6,
    }
)


## 1. Hyperparameter Tuning with Feature Subsets

For each `(model, time-series length, feature fraction)` combination:
- Build a symmetric feature subset
- Run Optuna hyperparameter search
- Log results to MLflow experiment `tuning_fs_1`
- Save best hyperparameters to `hyperparams/`

In [ ]:
def select_feature_indices(total_features: int, fraction: float) -> list[int]:
    """Select features symmetrically from both ends of the feature array."""
    count = max(1, round(total_features / 2 * fraction + 0.1))
    return sorted(
        set(range(count)) | set(range(total_features - count, total_features))
    )


for model in config.models:
    for data_dim in config.sample_sizes:
        data = pd.read_csv(
            f"{config.data_dir}/{config.train_data_subdir}/fbm_{data_dim}x{config.train_samples_num}.csv"
        )
        total_features = data.shape[1] - 1  # exclude target column

        for fraction in FEATURES_PERCENTAGES:
            print(f"Tuning {model} | dim={data_dim} | fraction={fraction}")

            feature_indices = select_feature_indices(total_features, fraction)
            X = data.drop(columns=[config.target_column]).iloc[:, feature_indices]
            y = data[config.target_column]

            estimator = get_model_by_name(
                model,
                random_state=config.random_seed,
                n_jobs=config.n_jobs,
                input_shape=(len(feature_indices), 1),
            )
            hyperparams_getter_func = get_hyperparams_getter_function_by_model_name(
                model
            )

            if model in ("RNN", "CNN"):
                X_train, X_val, y_train, y_val = train_test_split(
                    X,
                    y,
                    test_size=config.test_split_ratio,
                    random_state=config.random_seed,
                )
                study = tune_hyperparameters(
                    experiment_name=FS_EXPERIMENT_NAME,
                    run_name=f"{model}_{data_dim}_{fraction}",
                    estimator=estimator,
                    hyperparams_getter_func=hyperparams_getter_func,
                    scoring_func=root_mean_squared_error,
                    direction="minimize",
                    X_train=X_train,
                    y_train=y_train,
                    X_val=X_val,
                    y_val=y_val,
                    n_trials=config.trials[model],
                    cpus_to_use=config.n_jobs,
                )
            else:
                study = tune_hyperparameters(
                    experiment_name=FS_EXPERIMENT_NAME,
                    run_name=f"{model}_{data_dim}_{fraction}",
                    estimator=estimator,
                    hyperparams_getter_func=hyperparams_getter_func,
                    scoring_func=root_mean_squared_error,
                    direction="minimize",
                    X_train=X,
                    y_train=y,
                    cv_folds=config.cv_folds,
                    n_trials=config.trials[model],
                    cpus_to_use=config.n_jobs,
                )

            with open(f"hyperparams/{model}_{data_dim}.json", "w") as f:
                json.dump(
                    prepare_hyperparams(study.best_params, estimator), f, indent=4
                )

            print(f"  Done. Best RMSE: {study.best_value:.4f}")


## 2. RMSE vs Feature Fraction Analysis (LightGBM)

Load tuning results from MLflow and analyse how RMSE changes with feature fraction for LightGBM across all time-series lengths.

In [ ]:
def load_fs_results(experiment_name: str, model_name: str) -> pd.DataFrame:
    """Load best RMSE per (n_features, fraction) from an MLflow FS tuning experiment."""
    experiment = mlflow.get_experiment_by_name(experiment_name)
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        run_view_type=mlflow.entities.ViewType.ALL,
    )

    pattern = re.compile(rf"{model_name}_(\d+)_([01]\.\d+)")
    records = []
    for _, row in runs.iterrows():
        match = pattern.match(row.get("tags.mlflow.runName", ""))
        if not match:
            continue
        best_rmse = row.get("params.best_rmse") or row.get("metrics.best_rmse")
        if best_rmse is None:
            continue
        records.append(
            {
                "n_features": int(match.group(1)),
                "fraction": float(match.group(2)),
                "best_rmse": float(best_rmse),
            }
        )

    return pd.DataFrame(records)


df_fs = load_fs_results(FS_EXPERIMENT_NAME, "LightGBM")

# Pivot: rows = n_features, columns = fraction
pivot = df_fs.pivot_table(
    index="n_features", columns="fraction", values="best_rmse", aggfunc="min"
)
pivot


In [ ]:
def rmse_delta_table(df: pd.DataFrame, n_features: int) -> pd.DataFrame:
    """Return sorted table with RMSE and absolute delta vs best for a given n_features."""
    subset = (
        df[df["n_features"] == n_features]
        .drop(columns="n_features")
        .sort_values("fraction")
        .copy()
    )
    subset["delta_rmse"] = (subset["best_rmse"] - subset["best_rmse"].min()) * 100
    return subset


df_25 = rmse_delta_table(df_fs, 25)
df_50 = rmse_delta_table(df_fs, 50)
df_100 = rmse_delta_table(df_fs, 100)

print("=== 25 features ===")
display(df_25)
print("=== 50 features ===")
display(df_50)
print("=== 100 features ===")
display(df_100)


In [ ]:
def plot_rmse_vs_fraction(
    df: pd.DataFrame, n_features: int, save_path: str | None = None
):
    """Plot RMSE vs feature fraction with publication-quality styling."""
    fig, ax = plt.subplots(figsize=(5.5, 4.0))

    ax.plot(
        df["fraction"],
        df["best_rmse"],
        marker="o",
        linestyle="-",
        color="black",
        label=f"{n_features} features",
    )

    for _, row in df.iterrows():
        ax.annotate(
            f"{row['best_rmse']:.3f}",
            (row["fraction"], row["best_rmse"]),
            textcoords="offset points",
            xytext=(0, 6),
            ha="left",
            fontsize=9,
            clip_on=True,
        )

    ax.set_xlabel("Fraction of selected features")
    ax.set_ylabel("Best RMSE")
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
    ax.margins(x=0.1, y=0.15)
    ax.legend(frameon=False)
    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=300)

    return fig


plot_rmse_vs_fraction(df_25, 25, save_path="rmse_vs_fraction_25_features.png")
plot_rmse_vs_fraction(df_50, 50, save_path="rmse_vs_fraction_50_features.png")
plot_rmse_vs_fraction(df_100, 100, save_path="rmse_vs_fraction_100_features.png")
plt.show()
